# Temporal activity trends: calculation
This notebook calculates temporal trends of activity from the data ingested in 002 and saves one JSON file per person with their averages over follow-up.

## Set up your environment
Set your Cloud compute profile to 32 CPUs, 208 GB RAM and 500 GB Disk, with a highmem Compute type to get that configuration.

### Expected runtime
Approximate times on the recommended VM (32 CPUs, 208 GB RAM, highmem), measured on a run of 1,991 participants.

| Run | Participants | Approx. time |
|---|---|---|
| TEST = True | 50 participants | ~2 min |
| One chunk | ~1,000 participants | ~18 min |
| Full run | all participants ingested by 002 (~60 chunks) | ~18 h |

In [ ]:
%pip install ujson

In [ ]:
## Import libraries
import glob
import numpy as np
import h5py
import pandas as pd
import json
import time
from google.cloud import storage
import os
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import multiprocessing
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime, timedelta
from scipy import stats
import subprocess
import ujson

# WORKSPACE_CDR is not set automatically in this environment, so set it before anything reads it (WORKSPACE_BUCKET is set in the paths cell below)
os.environ["WORKSPACE_CDR"] = 'wb-silky-artichoke-2408.C2024Q3R9'

# Silence expected warnings from days or periods with no valid data (results are NaN there, which is intended)
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="After omitting NaNs")

NOTEBOOK_START = time.time()  # used for the total runtime printed at the end

def fmt_time(s):
    """Format seconds as h/min/s."""
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60}min {s % 60}s"


## Helper functions

In [ ]:
def fill_smart_nans(lst):

    """
    Intelligently fills NaN values in a list using contextual interpolation.

    Parameters:
    -----------
    lst : list
        Input list containing numeric values and NaN values that need to be filled.

    Returns:
    --------
    list
        A new list with all NaN values replaced using the following strategy:
        1. Leading NaNs: filled with mean of first 2 valid values
        2. Trailing NaNs: filled with mean of last 2 valid values
        3. Middle NaNs: filled with linear interpolation between neighboring valid values
    """

    arr = np.array(lst, dtype=float)
    isnan = np.isnan(arr)

    first_valid = np.argmax(~isnan)
    if first_valid > 0:
        first_two = arr[~isnan][:2]
        if len(first_two) >= 2:
            arr[:first_valid] = np.mean(first_two)

    last_valid = len(arr) - 1 - np.argmax(~isnan[::-1])
    if last_valid < len(arr) - 1:
        last_two = arr[~isnan][-2:]
        if len(last_two) >= 2:
            arr[last_valid + 1:] = np.mean(last_two)

    i = 0
    while i < len(arr):
        if np.isnan(arr[i]):
            start = i
            while i < len(arr) and np.isnan(arr[i]):
                i += 1
            end = i
            if start > 0 and end < len(arr):
                fill_val = (arr[start - 1] + arr[end]) / 2
                arr[start:end] = fill_val
        else:
            i += 1

    return arr.tolist()

def is_date_in_range(date_to_check, start_date, end_date):
    """
    Checks whether a date falls between start_date and end_date (inclusive).
    """
    date_to_check = pd.to_datetime(date_to_check)
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    return start_date <= date_to_check <= end_date

def compute_avg_mvg_per_complete_followup_year_irr_of_start_3_gap(followup_data, key):
    """
    Computes average, standard deviation, and median statistics for complete follow-up years
    with tolerance for missing data gaps, regardless of the starting point.

    Parameters:
    -----------
    followup_data : dict
        Nested dictionary structure: {key: {year: {month: [data_values]}}}
        Contains longitudinal data organized by year and month

    key : str
        The specific data type key to analyze (e.g., 'steps_limited', 'mvg_limited', 'hr_limited')

    Returns:
    --------
    tuple of 3 dictionaries: (avg_per_year, std_per_year, median_per_year)
        Each dictionary maps "Year N" → computed statistic for that follow-up year
        Returns empty dictionaries if there is no data
    """
    sorted_years = sorted(followup_data[key].keys())  # Ensure chronological order
    max_missing_months=3
    if not sorted_years:
        return {}, {}, {}  # No data

    # Extract earliest available month & year
    first_year = sorted_years[0]
    first_month = np.where(np.array([len(lst) for lst in followup_data[key][first_year].values()])>0)[0][0]+1 if \
         any(len(lst) for lst in followup_data[key][first_year].values()) else None
    if first_month is None:
        return {}, {}, {}  # No valid data

    # Initialize tracking
    step_counts_by_followup_year = defaultdict(list)
    year_index = 1
    current_start_year = first_year
    current_start_month = first_month

    while True:
        # Define the end month & year (rolling 12 months)
        end_date = datetime(current_start_year, current_start_month, 1) + timedelta(days=366)
        end_year = end_date.year
        end_month = end_date.month

        # Track data availability for the 12 month period
        full_year_data = []
        missing_months = 0
        month_tracker = (current_start_year, current_start_month)

        for _ in range(12):  # Loop through 12 months
            year, month = month_tracker
            has_data = False

            # Check if we have data for this month
            if year in followup_data[key] and month in followup_data[key][year]:
                if followup_data[key][year][month]:  # Ensure there is *some* data
                    full_year_data.extend(followup_data[key][year][month])
                    has_data = True

            # Count missing months
            if not has_data:
                missing_months += 1

            # Move to next month
            next_date = datetime(year, month, 1) + timedelta(days=32)
            month_tracker = (next_date.year, next_date.month)

        # Keep the year if no more than max_missing_months months are missing
        if missing_months <= max_missing_months and full_year_data:
            step_counts_by_followup_year[year_index] = full_year_data
            year_index += 1

            # Move to the next period
            current_start_year = end_year
            current_start_month = end_month
        else:
            break  # Stop processing when there's insufficient data for a year

    # Statistics per follow-up year
    avg_steps_per_year = {
        f"Year {i}": np.nanmean(step_counts_by_followup_year[i]) if step_counts_by_followup_year[i] else np.nan
        for i in range(1, year_index)
    }

    std_steps_per_year = {
        f"Year {i}": np.nanstd(step_counts_by_followup_year[i]) if step_counts_by_followup_year[i] else np.nan
        for i in range(1, year_index)
    }

    median_steps_per_year = {
        f"Year {i}": np.nanmedian(step_counts_by_followup_year[i]) if step_counts_by_followup_year[i] else np.nan
        for i in range(1, year_index)
    }

    return avg_steps_per_year, std_steps_per_year, median_steps_per_year

def rhr_fun(rhrarray):
    """
    Packages the daily resting heart rate (RHR) values with their count.

    Parameters:
    -----------
    rhrarray : list or array-like
        Array containing resting heart rate values
        Examples:
        - [65, 68, 62, 70, 66]  # Daily RHR values
        - [np.nan, 67, 65, np.nan, 69]  # RHR with missing days
        - []  # Empty array for no data

    Returns:
    --------
    dict
        Dictionary with the RHR values and their count:
        {
            "rhr_array": original_rhr_array,
            "rhr_len": number_of_elements_in_array
        }
    """
    return {
        "rhr_array": rhrarray,
        "rhr_len": len(rhrarray),
    }

def daily_stats_fun(array,full_type,activity_types):
    """
    Computes daily statistics (mean, std, median) for different activity types with
    flexible output format based on data dimensionality.

    Parameters:
    -----------
    array : dict
        Dictionary containing activity data arrays
        Structure: {activity_type: list_of_arrays or list_of_values}
        Examples:
        - Minute-level data: {'steps': [[day1_1440_values], [day2_1440_values], ...]}
        - Daily summary data: {'steps': [day1_total, day2_total, ...]}

    full_type : bool
        Controls the output format and expected input structure:
        - True: Expects minute-level data (1440 values per day), returns minute-level stats
        - False: Expects daily summary data (1 value per day), returns single summary stats

    activity_types : list
        List of activity type keys to process (e.g., 'steps_limited', 'mvg_limited', 'hr_limited')
        Must match keys present in the array dictionary

    Returns:
    --------
    dict
        Nested dictionary with computed statistics:
        {activity_type: {'non_nan_avg': values, 'non_nan_std': values, 'non_nan_median': values}}

        Output format depends on full_type:
        - If full_type=True: Each statistic is list of 1440 values (minute-by-minute averages)
        - If full_type=False: Each statistic is single value (overall summary)
    """
    if full_type:
        return {key: {
                "non_nan_avg": np.nanmean(array[key], axis=0).tolist() if array[key] else [np.nan] * 1440,
                "non_nan_std": np.nanstd(array[key], axis=0).tolist() if array[key] else [np.nan] * 1440,
                "non_nan_median": np.nanmedian(array[key], axis=0).tolist() if array[key] else [np.nan] * 1440,
            }
            for key in activity_types
        }
    else:
        return {key: {
                "non_nan_avg": np.nanmean(array[key], axis=0).tolist() if array[key] else np.nan,
                "non_nan_std": np.nanstd(array[key], axis=0).tolist() if array[key] else np.nan,
                "non_nan_median": np.nanmedian(array[key], axis=0).tolist() if array[key] else np.nan,
            }
            for key in activity_types
        }

def w_s_m_stats_fun(array,activity_types):
    """
    Computes Weekly, Seasonal, or Monthly statistics for activity data by calculating
    statistics for each time period (day/month) within the grouping.

    Parameters:
    -----------
    array : dict
        Nested dictionary containing activity data grouped by time periods
        Structure: {activity_type: {time_period: [list_of_values]}}

        Examples by context:
        - Weekly: {'steps': {'Monday': [1000, 1200, 900], 'Tuesday': [1100, 1300], ...}}
        - Monthly: {'steps': {1: [daily_values_for_1st], 2: [daily_values_for_2nd], ...}}
        - Seasonal: {'steps': {1: [jan_values], 2: [feb_values], ..., 12: [dec_values]}}

    activity_types : list
        List of activity type keys to process  (e.g., 'steps_limited', 'mvg_limited', 'hr_limited')
        Must match the outer keys in the array dictionary

    Returns:
    --------
    dict
        Nested dictionary with computed statistics for each time period:
        {activity_type: {
            'non_nan_avg': [avg_for_period1, avg_for_period2, ...],
            'non_nan_std': [std_for_period1, std_for_period2, ...],
            'non_nan_median': [median_for_period1, median_for_period2, ...]
        }}
    """
    return {
    key: {
        "non_nan_avg": [np.nanmean(array[key][day]) if array[key][day] else np.nan for day in array[key]],
        "non_nan_std": [np.nanstd(array[key][day]) if array[key][day] else np.nan for day in array[key]],
        "non_nan_median": [np.nanmedian(array[key][day]) if array[key][day] else np.nan for day in array[key]],
    }
    for key in activity_types
}

def yearly_stats_fun_3_gap(array,activity_types):
    """
    Computes yearly statistics for longitudinal activity data using a specialized
    follow-up year analysis that tolerates up to 3 months of missing data per year.

    Parameters:
    -----------
    array : dict
        Nested dictionary containing longitudinal activity data
        Structure: {activity_type: {year: {month: [daily_values]}}}
        Example: {
            'steps': {
                2023: {1: [8000, 8200, 7800], 2: [8500, 8300], 3: [], ...},
                2024: {1: [8100, 8400], 2: [8600, 8200], ...}
            }
        }

    activity_types : list
        List of activity type keys to process (e.g., 'steps_limited', 'mvg_limited', 'hr_limited')
        Must match the outer keys in the array dictionary

    Returns:
    --------
    dict
        Nested dictionary with yearly statistics for each activity type:
        {activity_type: {
            'non_nan_avg': [dict_of_yearly_averages],
            'non_nan_std': [dict_of_yearly_std_devs],
            'non_nan_median': [dict_of_yearly_medians]
        }}

        Each inner list contains a single dictionary mapping follow-up years to values:
        Example: {'non_nan_avg': [{'Year 1': 8234.5, 'Year 2': 8567.2}]}
    """
    return {
    key: {
        "non_nan_avg": [compute_avg_mvg_per_complete_followup_year_irr_of_start_3_gap(array, key)[0]],
        "non_nan_std": [compute_avg_mvg_per_complete_followup_year_irr_of_start_3_gap(array, key)[1]],
        "non_nan_median": [compute_avg_mvg_per_complete_followup_year_irr_of_start_3_gap(array, key)[2]],
    }
    for key in activity_types
}

## Calculate activity trends

In [ ]:
def process_hr_mvg_sd_file(qc_file, dob, df_summary_hr, vl, dl, wk, \
                        mn, ss, yy, rhr_dt):

    """
    Processes one person's heart rate, step and sleep data from their HDF5 file, classifies each
    minute as moderate to vigorous, light or sedentary activity, and groups the daily values
    (activity minutes, steps, heart rate) across several time scales.

    Parameters:
    -----------
    qc_file : str
        Path to HDF5 file containing quality-controlled heart rate and step data
        Structure: {date_string: {'hr_array_qc': array, 'steps_array_qc': array, 'sleep_array': array}}

    dob : datetime
        Date of birth for calculating age-adjusted maximum heart rate (220 - age formula)

    df_summary_hr : pandas.DataFrame
        DataFrame with each person's heart rate follow-up:
        - 'person_id': participant identifier
        - 'longest_streak_start': start date of longest continuous data period
        - 'longest_streak_end': end date of longest continuous data period

    vl, dl, wk, mn, ss, yy : dict
        Pre-initialized nested dictionaries for storing aggregated data:
        - vl: Daily summary values {activity_type: [daily_totals]}
        - dl: Daily minute-level arrays {activity_type: [1440_minute_arrays]}
        - wk: Weekly aggregation {activity_type: {day_name: [values]}}
        - mn: Monthly aggregation {activity_type: {day_of_month: [values]}}
        - ss: Seasonal aggregation {activity_type: {month: [values]}}
        - yy: Yearly aggregation {activity_type: {year: {month: [values]}}}

    rhr_dt : list
        List to store resting heart rate values for each processed day

    Returns:
    --------
    tuple
        Updated versions of all input data structures: (vl, dl, wk, mn, ss, yy, rhr_dt)
    """

    person_id = os.path.basename(qc_file).replace(".hd5", "")
    if os.path.exists(qc_file):

        # Seasonal and yearly trends only use days inside the person's longest continuous follow-up streak
        person_hr_data = df_summary_hr[df_summary_hr['person_id']==person_id]
        has_streak_data = len(person_hr_data) > 0
        if has_streak_data:
            streak_s_hr = person_hr_data['longest_streak_start'].values[0]
            streak_e_hr = person_hr_data['longest_streak_end'].values[0]

        with h5py.File(qc_file, 'r') as a:
            for jjk in range(len(sorted(a.keys()))):
                missing_sleep_flag = False
                len_mismatch = False
                date_str = sorted(a.keys())[jjk]
                ref_date = datetime.strptime(date_str[0:10], "%Y-%m-%d")
                age = ref_date.year - dob.year - ((ref_date.month, ref_date.day) < (dob.month, dob.day))
                max_hr_ref = 220 - age

                date_obj = pd.to_datetime(date_str)
                day_of_week = date_obj.day_name()
                day_of_month = date_obj.day  # Day of the month (1-31)
                month = date_obj.month  # Returns an integer (1 for January, 2 for February, ..., 12 for December)
                year = date_obj.year

                hr_array = a[date_str]['hr_array_qc'][()]
                steps_array = a[date_str]['steps_array_qc'][()]
                sleep_array = a[date_str]['sleep_array'][()]

                in_streak = (has_streak_data and
                        is_date_in_range(str(date_str)[0:10], str(streak_s_hr)[0:10], str(streak_e_hr)[0:10]))

                # Skip days with neither heart rate nor step data
                if hr_array.size == 0 and steps_array.size == 0:
                    continue

                # With both heart rate and step data, classify each minute of the day
                if hr_array.size > 0 and steps_array.size > 0:
                    step_nw = steps_array.copy().astype(float)
                    step_nw[np.isnan(hr_array)] = np.nan

                    if sleep_array.size > 0:
                        sleep_array[~np.isnan(sleep_array)] = 1
                        sleep_array[np.isnan(sleep_array)] = 0
                    else:
                        sleep_array = np.full(len(hr_array), 0)

                    if np.sum(sleep_array) == 0:
                        missing_sleep_flag = True

                    if len(sleep_array) == len(hr_array):
                        heart_rate_non_sleep = np.where(sleep_array == 0, hr_array, np.nan)
                        heart_rate_sleep     = np.where(sleep_array == 1, hr_array, np.nan)
                        steps_non_sleep      = np.where(sleep_array == 0, steps_array, np.nan)
                        steps_sleep          = np.where(sleep_array == 1, steps_array, np.nan)
                        steps_nw_non_sleep   = np.where(sleep_array == 0, step_nw, np.nan)
                        steps_nw_sleep       = np.where(sleep_array == 1, step_nw, np.nan)
                    else:
                        len_mismatch = True
                        continue  # sleep array doesn't line up with heart rate/steps, so skip the day

                    steps_missingness_flag_1240 = np.sum(np.isnan(step_nw)) >= 200
                    hr_missingness_flag_1240 = np.sum(np.isnan(hr_array)) >= 200

                    # Resting heart rate (RHR) is taken as the most common (mode) heart rate of the day
                    RHR = stats.mode(hr_array, nan_policy='omit').mode
                    RHR_sleep = stats.mode(heart_rate_sleep, nan_policy='omit').mode
                    RHR_non_sleep = stats.mode(heart_rate_non_sleep, nan_policy='omit').mode

                    hr_clean = heart_rate_non_sleep.astype(float)
                    st_clean = steps_nw_non_sleep.astype(float)

                    # Heart rate reserve = max heart rate (220 - age) - RHR. Only awake minutes are classified.
                    # MVPA: heart rate at or above RHR + 40% of the reserve, with steps > 0
                    # Light activity: between RHR + 20% and RHR + 40% of the reserve, with steps > 0
                    # Sedentary: at or below RHR + 20% of the reserve, with zero steps

                    max_mvg_ref_mvg = (40 * (max_hr_ref - RHR)) / 100
                    max_mvg_ref_sd  = (20 * (max_hr_ref - RHR)) / 100

                    hr_array_cp_mvg = np.where(np.isnan(hr_clean), np.nan, \
                                                  ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean > 0)).astype(float))
                    hr_array_cp_lact = np.where(np.isnan(hr_clean), np.nan, \
                             ((hr_clean > (max_mvg_ref_sd+RHR)) & (hr_clean < (max_mvg_ref_mvg+RHR)) & (st_clean > 0)).astype(float))
                    hr_array_cp_sd = np.where(np.isnan(hr_clean), np.nan,\
                                                  ((hr_clean <= (max_mvg_ref_sd+RHR)) & (st_clean == 0)).astype(float))

                    mvg_sum = np.nansum(hr_array_cp_mvg)

                    sd_sum = np.nansum(hr_array_cp_sd)

                    lact_sum = np.nansum(hr_array_cp_lact)

                    rhr_dt.append(RHR)

                    # Only keep days with fewer than 200 missing minutes of heart rate and steps, and with sleep data
                    if (not hr_missingness_flag_1240) and (not steps_missingness_flag_1240) and (not missing_sleep_flag):
                        vl['mvg_limited'].append(mvg_sum)
                        dl['mvg_limited'].append(hr_array_cp_mvg)
                        vl['sd_limited'].append(sd_sum)
                        dl['sd_limited'].append(hr_array_cp_sd)
                        vl['lact_limited'].append(lact_sum)
                        dl['lact_limited'].append(hr_array_cp_lact)

                        wk['mvg_limited'][day_of_week].append(mvg_sum)
                        mn['mvg_limited'][day_of_month].append(mvg_sum)
                        wk['sd_limited'][day_of_week].append(sd_sum)
                        mn['sd_limited'][day_of_month].append(sd_sum)
                        wk['lact_limited'][day_of_week].append(lact_sum)
                        mn['lact_limited'][day_of_month].append(lact_sum)

                        if in_streak:
                            ss['mvg_limited'][month].append(mvg_sum)
                            yy['mvg_limited'][year][month].append(mvg_sum)
                            ss['sd_limited'][month].append(sd_sum)
                            yy['sd_limited'][year][month].append(sd_sum)
                            ss['lact_limited'][month].append(lact_sum)
                            yy['lact_limited'][year][month].append(lact_sum)

                # Daily step total and mean heart rate, with the same day filters as above
                if hr_array.size > 0 and steps_array.size > 0:
                    steps_sum = np.nansum(step_nw)
                    hr_m = np.nanmean(hr_array)

                    if (not hr_missingness_flag_1240) and (not steps_missingness_flag_1240) and (not missing_sleep_flag):
                        vl['steps_limited'].append(steps_sum)
                        dl['steps_limited'].append(step_nw)
                        wk['steps_limited'][day_of_week].append(steps_sum)
                        mn['steps_limited'][day_of_month].append(steps_sum)

                        vl['hr_limited'].append(hr_m)
                        dl['hr_limited'].append(hr_array)
                        wk['hr_limited'][day_of_week].append(hr_m)
                        mn['hr_limited'][day_of_month].append(hr_m)

                        if in_streak:
                            ss['steps_limited'][month].append(steps_sum)
                            yy['steps_limited'][year][month].append(steps_sum)
                            ss['hr_limited'][month].append(hr_m)
                            yy['hr_limited'][year][month].append(hr_m)

    return vl, dl, wk, mn, ss, yy, rhr_dt

In [ ]:
def process_mvg_sd_calculation(qc_file,df_summary_i,df_summary_hr):

    """
    Runs the trend calculation for one person and packages the results for saving as JSON.

    Parameters:
    -----------
    qc_file : str
        Path to HDF5 file containing quality-controlled heart rate and step data
        Structure: {date_string: {'hr_array_qc': array, 'steps_array_qc': array, 'sleep_array': array}}

    df_summary_i : pandas.DataFrame
        DataFrame with each person's date of birth

    df_summary_hr : pandas.DataFrame
        DataFrame with each person's heart rate follow-up:
        - 'person_id': participant identifier
        - 'longest_streak_start': start date of longest continuous data period
        - 'longest_streak_end': end date of longest continuous data period

    Returns:
    --------
    tuple
        (json_data, output_path): the person's daily, weekly, monthly, seasonal and yearly
        statistics as a dictionary, and the path of the JSON file to save it to
    """

    # Parallel workers do not inherit the notebook's warning filters, so set them here too
    import warnings
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    warnings.filterwarnings("ignore", message="After omitting NaNs")

    person_id = os.path.basename(qc_file).replace(".hd5", "")
    dob_str = df_summary_i[df_summary_i['person_id']==int(person_id)]['date_of_birth'].values[0][0:10]
    dob = datetime.strptime(dob_str, "%Y-%m-%d")

    activity_types = ['mvg_limited','sd_limited',\
                      'lact_limited','steps_limited','hr_limited']

    all_act_lvl = {key: [] for key in activity_types}
    all_min_act_lvl = {key: [] for key in activity_types}
    weekly_act_lvl = {key: {day: [] for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]} for key in activity_types}
    monthly_act_lvl = {key: {day: [] for day in range(1, 32)} for key in activity_types}  # values for each day of the month, across all months
    seasonal_act_lvl = {key: {month: [] for month in range(1, 13)} for key in activity_types}
    yearly_act_lvl = { key: defaultdict(lambda: {month: [] for month in range(1, 13)}) for key in activity_types}
    rhr_act_lvl = []

    all_act_lvl, all_min_act_lvl, weekly_act_lvl, \
monthly_act_lvl, seasonal_act_lvl, yearly_act_lvl, rhr_act_lvl = process_hr_mvg_sd_file(qc_file, dob, \
                            df_summary_hr, all_act_lvl, all_min_act_lvl, \
                       weekly_act_lvl, monthly_act_lvl, seasonal_act_lvl, yearly_act_lvl, rhr_act_lvl)

    daily_one_value_act_lvl = daily_stats_fun(all_act_lvl,False,activity_types)
    daily_array_act_lvl = daily_stats_fun(all_min_act_lvl,True,activity_types)
    daily_rhr_array_act_lvl = rhr_fun(rhr_act_lvl)

    weekly_stats_act_lvl = w_s_m_stats_fun(weekly_act_lvl,activity_types)
    seasonal_stats_act_lvl = w_s_m_stats_fun(seasonal_act_lvl,activity_types)
    monthly_stats_act_lvl = w_s_m_stats_fun(monthly_act_lvl,activity_types)
    yearly_stats_act_lvl_3_gap = yearly_stats_fun_3_gap(yearly_act_lvl,activity_types)

    # JSON structure saved per person (read by 005)
    json_data = {
        "person_id": person_id,
        "daily_one_value_act_lvl": daily_one_value_act_lvl,
        "daily_array_act_lvl": daily_array_act_lvl ,
        "daily_rhr_array_act_lvl": daily_rhr_array_act_lvl,
        "weekly_stats_act_lvl": weekly_stats_act_lvl,
        "seasonal_stats_act_lvl": seasonal_stats_act_lvl,
        "monthly_stats_act_lvl": monthly_stats_act_lvl,
        "yearly_stats_act_lvl_3_gap": yearly_stats_act_lvl_3_gap,
    }

    return json_data, f"jsons_calculations/{person_id}.json"

## Run the calculation
The QC files from 002 are copied to the VM, and every participant is processed in parallel (one process per CPU). Each participant's results are saved as a JSON file and uploaded to `{trends_save}/jsons_calculations/` in the workspace bucket, where 005 reads them.

The notebook uses whichever 002 outputs exist: the EHR group, the no-EHR group, or both. A TEST run only has the EHR group.

In [ ]:
# --- PRODUCTION: workspace's own bucket (works in any copy of the workspace) ---
BUCKET  = subprocess.check_output(['wb', 'resource', 'resolve', '--id=multimodal_risk_prediction']).decode().strip() #gs bucket path
bucket_ = '/home/jupyter/workspace/multimodal_risk_prediction' #local fuse mounted path
os.environ["WORKSPACE_BUCKET"] = BUCKET

# Folder and file names used by this notebook (from filepaths_values.csv)
file_paths_values = pd.read_csv('filepaths_values.csv')
file_paths_values = file_paths_values[file_paths_values['notebook']=='trends_calculation'][['notebook','key_','value_']]
paths_dict = file_paths_values.set_index("key_")["value_"]

# TEST = True: quick end-to-end test. 001 pulls only the first 50 EHR participants (chunk 1), and every folder/file name
# from filepaths_values.csv gets a '_test' suffix in 001-005, so a test never overwrites a full run.
# TEST is set once in 001 and read here (run 001's setup cell first).
TEST_FLAG_FILE = f'{bucket_}/test_flag.txt'
assert os.path.exists(TEST_FLAG_FILE), "TEST flag not found: run the setup cell in 001 first, it saves TEST for 002-005"
TEST = open(TEST_FLAG_FILE).read().strip() == 'True'
if TEST:
    paths_dict = paths_dict.map(lambda v: v if '.' in str(v) else f'{v}_test')
print(f"TEST={TEST}", dict(paths_dict))

In [ ]:
# Date of birth per person (used for age-adjusted max heart rate)
df_summary_i = pd.read_csv(f'{bucket_}/{paths_dict.get("person_stats_file")}.tsv', sep='\t')

# Heart rate follow-up per person:
# - 'person_id': participant identifier
# - 'longest_streak_start': start date of the longest continuous data period
# - 'longest_streak_end': end date of the longest continuous data period
df_summary_hr = pd.read_csv(f'{bucket_}/{paths_dict.get("followup_heartrate_file")}.tsv', sep='\t')
df_summary_hr['person_id'] = df_summary_hr['person_id'].astype(str)

folder_to_save = paths_dict.get("trends_save")

# Use whichever groups 002 has written QC files for: EHR, no-EHR, or both
def has_qc_files(folder):
    r = subprocess.run(f"gcloud storage ls '{BUCKET}/{folder}/chunk_*_qc/*.hd5'", shell=True, capture_output=True)
    return r.returncode == 0  # gcloud returns non-zero when nothing matches

ingest_folders = [paths_dict.get(k) for k in ["folder_ingest_ehr", "folder_ingest_no_ehr"] if has_qc_files(paths_dict.get(k))]
assert ingest_folders, "No QC files found from 002: run 002 first"
print("Calculating trends on:", ingest_folders)

# Local working folders; clear leftovers from an earlier run so they are not processed or uploaded again
qc_folder = "all_qc_local"
subprocess.run(f"rm -rf {qc_folder} jsons_calculations", shell=True)
os.makedirs(qc_folder, exist_ok=True)
os.makedirs("jsons_calculations", exist_ok=True)

timings = {}
t0 = time.time()

# Copy the QC files written by 002 (one chunk_{n}_qc folder per chunk) from the gs:// bucket; gcloud cannot copy from the mounted folder
for folder in ingest_folders:
    subprocess.run(f"gcloud storage cp '{BUCKET}/{folder}/chunk_*_qc/*.hd5' {qc_folder}/", shell=True, check=True)

timings["Copy QC files from bucket"] = time.time() - t0

qc_files = sorted(glob.glob(f"{qc_folder}/*.hd5"))
print(f"{len(qc_files)} participants to process")

num_cores = multiprocessing.cpu_count()

t0 = time.time()
json_data_v = Parallel(n_jobs=num_cores)(delayed(process_mvg_sd_calculation)(qc_file, df_summary_i, df_summary_hr) for qc_file in tqdm(qc_files))
timings["Trend calculation"] = time.time() - t0

def write_json_file(json_data, filename):
    """Write one person's results to a JSON file."""
    with open(filename, "w") as json_file:
        json_file.write(ujson.dumps(json_data))

t0 = time.time()
Parallel(n_jobs=num_cores)(delayed(write_json_file)(json_v, name) for json_v, name in json_data_v)
timings["Write JSON files"] = time.time() - t0

# Upload to {trends_save}/jsons_calculations/ in the gs:// bucket (read by 005), then remove the local QC copies
t0 = time.time()
subprocess.run(f"gcloud storage cp -r jsons_calculations {BUCKET}/{folder_to_save}/", shell=True, check=True)
subprocess.run(f"rm -r {qc_folder}", shell=True, check=True)
timings["Upload JSON files"] = time.time() - t0

print(f"\nTimings for {len(qc_files)} participants:")
for step, secs in timings.items():
    print(f"  {step:<28} {fmt_time(secs)}")

In [ ]:
print(f"Total runtime of 004: {fmt_time(time.time() - NOTEBOOK_START)}")

## Next steps
Continue in `005_v8_trends_figures_release`, which builds the temporal trend figures from the calculations saved here.